# SEAS 8515 – Data Engineering for AI
## Homework 3

### Question 1: Data Ingestion and Transformation

#### Part A: Load and Convert

In [ ]:
import pandas as pd

# Load CSV into DataFrame
df = pd.read_csv('air_quality.csv')
print("Shape:", df.shape)
print(df.head())

# Save as Parquet using PyArrow engine
df.to_parquet('air_quality.parquet', engine='pyarrow', index=False)
print("\nSaved air_quality.parquet")

#### Part B: Clean the Data

In [ ]:
import pandas as pd

# Load Parquet
df = pd.read_parquet('air_quality.parquet', engine='pyarrow')

# Check missing values
print("Missing values before cleaning:")
print(df.isnull().sum())

# Drop predominantly empty columns (sunshine minutes is >99% missing)
threshold = 0.5
df = df.dropna(thresh=int(len(df) * threshold), axis=1)
print("\nColumns after dropping predominantly empty:", df.columns.tolist())

# Drop rows with remaining missing values
df = df.dropna()
print(f"\nShape after dropping NaN rows: {df.shape}")

# Standardize column names: lowercase and replace spaces with underscores
df.columns = df.columns.str.lower().str.replace(' ', '_')
print("Standardized columns:", df.columns.tolist())

# Show statistics
print("\nDataFrame statistics:")
print(df.describe())

# Save cleaned DataFrame
df.to_parquet('air_quality_cleaned.parquet', engine='pyarrow', index=False)
print("\nSaved air_quality_cleaned.parquet")

#### Part C: Feature Engineering and Filtering

In [ ]:
import pandas as pd

# Load cleaned Parquet
df = pd.read_parquet('air_quality_cleaned.parquet', engine='pyarrow')

# Extract California subset (aqs_id first two characters == '06')
air_quality_california = df[df['aqs_id'].str[:2] == '06'].copy()
print(f"California rows: {air_quality_california.shape}")

# Convert temperature from Celsius to Fahrenheit
air_quality_california['temp_fahrenheit'] = air_quality_california['avg_temp_centrigrade'] * 9 / 5 + 32

# Show statistics
print("\nCalifornia DataFrame statistics:")
print(air_quality_california.describe())

#### Part D: Final Report

In [ ]:
import pandas as pd
import plotly.express as px

# Parse date column
air_quality_california['date_local'] = pd.to_datetime(air_quality_california['date_local'])

# Filter for 2020
ca_2020 = air_quality_california[air_quality_california['date_local'].dt.year == 2020]

# Top 3 sensor locations by max PM2.5
top3 = (
    ca_2020.groupby(['aqs_id', 'latitude_x', 'longitude_x'])
    .agg(max_pm25=('pm2.5_conc', 'max'))
    .reset_index()
    .nlargest(3, 'max_pm25')
)

print("Top 3 California sensor locations (2020) by max PM2.5:")
print(top3)

# Plot on a tile map zoomed to California
fig = px.scatter_map(
    top3,
    lat='latitude_x',
    lon='longitude_x',
    size='max_pm25',
    color='max_pm25',
    text='aqs_id',
    hover_data={'aqs_id': True, 'max_pm25': True, 'latitude_x': True, 'longitude_x': True},
    title='Top 3 California Sensor Locations with Highest PM2.5 Concentrations (2020)',
    color_continuous_scale='YlOrRd',
    range_color=[0, top3['max_pm25'].max()],
    map_style='open-street-map',
    center={'lat': 37.0, 'lon': -119.5},
    zoom=4.5,
    size_max=30,
)
fig.show()

### Question 2: The Four Vs of Big Data

Provide written responses below:

- **Volume:** The sheer scale of data generated by modern systems presents significant storage and processing challenges. A single IoT sensor network like the one in this dataset can accumulate millions of records over just a few years, and at enterprise scale this grows to petabytes or more. Traditional relational databases struggle under this load, often becoming bottlenecks for both ingestion and query performance. Mitigation strategies include adopting columnar storage formats like Parquet (as used in this assignment) to reduce I/O, leveraging distributed compute frameworks like Spark or Dask, and tiering data across hot and cold storage based on access frequency. Data lifecycle policies—archiving or deleting records past a retention window—also help keep volume manageable over time.

- **Velocity:** Velocity refers to the speed at which data is generated, ingested, and must be acted upon. Real-time sensor data, financial tick feeds, and social media streams can arrive at millions of events per second, far outpacing what batch pipelines can handle. If processing lags behind ingestion, queues back up, latency grows, and downstream consumers receive stale signals. Mitigation strategies include stream processing frameworks like Apache Kafka and Apache Flink, which decouple ingestion from processing and allow horizontal scaling of consumers. Designing pipelines with backpressure handling and circuit breakers also prevents cascading failures when velocity spikes unexpectedly.

- **Variety:** Data today arrives in a wide range of formats—structured tables, semi-structured JSON and XML, unstructured text, images, audio, and binary sensor payloads—all of which may need to be combined for a single analysis. This heterogeneity makes schema management difficult, since different sources evolve their formats independently and at different speeds. Integrating these sources requires significant transformation work, often involving custom parsers, schema registries, and data contracts between teams. Mitigation strategies include adopting a lakehouse architecture that stores raw data in its native format and applies schema-on-read, and using tools like Apache Avro or Protobuf to enforce contracts at the producer level. Investing in a metadata catalog also makes it easier to discover and understand what variety of data is available across an organization.

- **Veracity:** Veracity addresses the trustworthiness and accuracy of data—whether it reflects reality, is free from errors, and can be relied upon for decision-making. The air quality dataset in this assignment illustrates the problem well: the `sunshine_minutes` column was over 99% missing, and other columns had scattered nulls that could silently skew downstream models if not addressed. Data quality issues can stem from faulty sensors, manual entry errors, upstream system changes, or deliberate manipulation. Mitigation strategies include automated data quality checks at ingestion (null rate monitoring, range validation, referential integrity checks), lineage tracking so errors can be traced to their source, and robust imputation or flagging strategies for missing values. Establishing data ownership and SLAs with upstream providers also creates accountability for quality over time.

**Biggest Challenge:** Of the four Vs, I consider Veracity to be the most difficult to address in practice. Volume and Velocity are ultimately engineering problems—throw enough compute and storage at them and they can be solved, at least temporarily. Variety is a design challenge that can be managed through good architecture and tooling choices. Veracity, however, is fundamentally a trust problem: bad data can flow silently through every layer of a pipeline and only reveal itself when a model produces a wrong prediction or a business decision goes sideways. Unlike a crashing system, corrupt data rarely throws an error—it just quietly produces wrong answers, making it the hardest category to detect and the most expensive to fix after the fact.

**Most Common in Your Experience:** In my professional experience, I most frequently encounter challenges related to Variety and Veracity. Systems I have worked with routinely ingest data from multiple upstream sources that have different schemas, naming conventions, and update cadences, requiring constant normalization work. Veracity issues surface almost as often—data that looks correct passes through pipelines, only for analysts to discover inconsistencies when comparing results across sources. Volume is a factor but tends to be more predictable and addressable with infrastructure investment. Velocity is the least common challenge in my day-to-day work, as most of the pipelines I interact with are batch-oriented rather than real-time.